In [20]:
import pandas as pd
import geopandas as gpd
import openpyxl

In [21]:
# --- Load base crime dataset ---
crime = pd.read_csv(r"C:\Users\kotha\Downloads\DTSC-3602-Group-12-Project\data\Crime_Data_from_2020_to_Present.csv")

# --- Load the shapefile ---
tracts = gpd.read_file(r"C:\Users\kotha\Downloads\DTSC-3602-Group-12-Project\data\2020_Census_Tracts\2020_Census_Tracts.shp")

In [22]:
# Convert crime data to GeoDataFrame
crime_gdf = gpd.GeoDataFrame(
  crime,
  geometry=gpd.points_from_xy(crime["LON"], crime["LAT"]),
  crs="EPSG:4326"
)

In [23]:
tracts.columns

Index(['OBJECTID', 'CT20', 'LABEL', 'ShapeSTAre', 'ShapeSTLen', 'geometry'], dtype='object')

In [24]:
# --- Ensure both layers use the same coordinate system ---
print("Crime CRS:", crime_gdf.crs)
print("Tracts CRS:", tracts.crs)

# Convert tracts to match the crime data CRS
tracts = tracts.to_crs(crime_gdf.crs)

Crime CRS: EPSG:4326
Tracts CRS: EPSG:2229


In [25]:
# --- Spatial join: assign Census Tract CT20 to each crime ---
crime_with_tract = gpd.sjoin(
  crime_gdf,
  tracts[["CT20", "geometry"]],
  how="left",
  predicate="within"
).rename(columns={"CT20": "geoid20"})

In [26]:
crime_with_tract["geoid20"].isna().mean()

0.0022288756814737645

In [27]:
# --- Load external datasets ---
econ = pd.read_csv(r"C:\Users\kotha\Downloads\DTSC-3602-Group-12-Project\data\i16_Census_Tract_EconomicallyDistressedAreas_2023.csv")
income = pd.read_csv(r"C:\Users\kotha\Downloads\DTSC-3602-Group-12-Project\data\All Years Median Household Income Calculations.csv")
vehicles = pd.read_csv(r"C:\Users\kotha\Downloads\DTSC-3602-Group-12-Project\data\All Years Vehicle Ownership Calculations.csv")
htc = pd.read_excel(r"C:\Users\kotha\Downloads\DTSC-3602-Group-12-Project\data\Tract-level-CA-HTC-Index-for-public-website-download-20190304.xlsx")

In [29]:
# --- Clean merge keys ---
for df in [econ, income, vehicles, htc]:
  df.columns = df.columns.str.lower()

In [32]:
# Ensure GEOID fields are consistent in type and length
econ["geoid20"] = econ["geoid20"].astype(str)
income["geoid20"] = income["geoid20"].astype(str)
vehicles["geoid20"] = vehicles["geoid20"].astype(str)
htc["geoid"] = htc["geoid"].astype(str)

crime_with_tract["geoid20"] = crime_with_tract["geoid20"].astype(str)

In [34]:
print(htc.columns)
htc.head()

Index(['geoid', 'ca htc index'], dtype='object')


,geoid,ca htc index
0,6001400100,20
1,6001400200,16
2,6001400300,31
3,6001400400,35
4,6001400500,47


In [35]:
# Clean up column names (make them uniform)
htc.columns = htc.columns.str.strip().str.replace(' ', '_').str.replace('-', '_')

# Now check again
print(htc.columns)

Index(['geoid', 'ca_htc_index'], dtype='object')


In [37]:
# --- Merge external data with Census GEOID ---
merged = (
    crime_with_tract
        .merge(econ[["geoid20", "mhi23", "pop23", "popdensity"]], on="geoid20", how="left")
        .merge(income[["geoid20", "med_hh_inc_adj"]], on="geoid20", how="left")
        .merge(vehicles[["geoid20", "no_vehicle_pct"]], on="geoid20", how="left")
        .merge(htc[["geoid", "ca_htc_index"]].rename(columns={"geoid": "geoid20"}), on="geoid20", how="left")
)

merged

,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,...,LON,geometry,index_right,geoid20,mhi23,pop23,popdensity,med_hh_inc_adj,no_vehicle_pct,ca_htc_index
0,211507896,04/11/2021 12:00:00 AM,11/07/2020 12:00:00 AM,845,15,N Hollywood,1502,2,354,THEFT OF IDENTITY,...,-118.4092,POINT (-118.4092 34.2124),174.0,121600,NaN,NaN,NaN,NaN,NaN,NaN
1,201516622,10/21/2020 12:00:00 AM,10/18/2020 12:00:00 AM,1845,15,N Hollywood,1521,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",...,-118.4203,POINT (-118.4203 34.1993),196.0,123410,NaN,NaN,NaN,NaN,NaN,NaN
2,240913563,12/10/2024 12:00:00 AM,10/30/2020 12:00:00 AM,1240,9,Van Nuys,933,2,354,THEFT OF IDENTITY,...,-118.4509,POINT (-118.4509 34.1847),258.0,128303,NaN,NaN,NaN,NaN,NaN,NaN
3,210704711,12/24/2020 12:00:00 AM,12/24/2020 12:00:00 AM,1310,7,Wilshire,782,1,331,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,...,-118.3747,POINT (-118.3747 34.0339),982.0,270200,NaN,NaN,NaN,NaN,NaN,NaN
4,201418201,10/03/2020 12:00:00 AM,09/29/2020 12:00:00 AM,1830,14,Pacific,1454,1,420,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),...,-118.4350,POINT (-118.435 33.9813),1015.0,275312,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1004986,252104112,02/02/2025 12:00:00 AM,02/02/2025 12:00:00 AM,130,21,Topanga,2103,2,946,OTHER MISCELLANEOUS CRIME,...,-118.6126,POINT (-118.6126 34.2259),99.0,113232,NaN,NaN,NaN,NaN,NaN,NaN
1004987,250404100,02/18/2025 12:00:00 AM,02/18/2025 12:00:00 AM,1000,4,Hollenbeck,479,2,237,CHILD NEGLECT (SEE 300 W.I.C.),...,-118.1979,POINT (-118.1979 34.0277),573.0,204810,NaN,NaN,NaN,NaN,NaN,NaN
1004988,251304095,01/31/2025 12:00:00 AM,01/30/2025 12:00:00 AM,1554,13,Newton,1372,2,850,INDECENT EXPOSURE,...,-118.2701,POINT (-118.2701 33.9942),796.0,229410,NaN,NaN,NaN,NaN,NaN,NaN
1004989,251704066,01/17/2025 12:00:00 AM,01/17/2025 12:00:00 AM,1600,17,Devonshire,1774,2,624,BATTERY - SIMPLE ASSAULT,...,-118.5233,POINT (-118.5233 34.245),117.0,115103,NaN,NaN,NaN,NaN,NaN,NaN
